# TFT Test Set Evaluation — Cáceres Solar Forecast

This notebook evaluates the trained TFT model on the held-out test set (2025-07-01 to 2025-12-31).
The test set is touched **exactly once** here. No model selection or hyperparameter decisions are
made in this notebook.

**MLVU evaluation principle:** the test set is a simulation of production deployment. Any information
available during evaluation that wouldn't exist in production constitutes leakage.

**Pipeline steps:**

| # | Step | Purpose |
|---|---|---|
| 0 | Imports & configuration | Libraries, load preprocessing and training params |
| 1 | Reconstruct dataset for test split | Continuous time_idx across all splits |
| 2 | Load model & generate predictions | Best checkpoint from training |
| 3 | Test metrics | RMSE, MAE, R², MASE on original MWh scale |
| 4 | Baseline comparisons | Persistence (24h), climatological (hourly mean) |
| 5 | Visualizations | Time series, scatter, residuals, TFT interpretability |
| 6 | Summary | Final results table |

## Step 0 — Imports & configuration

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import lightning.pytorch as pl

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

import functools
_orig_torch_load = torch.load
@functools.wraps(_orig_torch_load)
def _patched_torch_load(*args, **kwargs):
    if kwargs.get('weights_only') is None:
        kwargs['weights_only'] = False
    return _orig_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# ── Paths ──
DATA_DIR  = os.path.join(os.getcwd(), 'data')
MODEL_DIR = os.path.join(os.getcwd(), 'models')

# ── Sequence lengths ──
MAX_ENCODER_LENGTH    = 168
MAX_PREDICTION_LENGTH = 24

# ── Feature classification ──
KNOWN_FUTURE = [
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'doy_sin', 'doy_cos', 'solar_zenith', 'solar_azimuth', 'clearsky_ghi',
]
OBSERVED = [
    'dewpoint_2m_C', 'temperature_2m_C', 'surface_pressure_hPa',
    'total_precip_mm', 'ssrd_wm2', 'strd_wm2', 'kt', 'dewpoint_depression_C',
]
TARGET = 'pv_generation_mwh'

# ── Load params ──
with open(os.path.join(DATA_DIR, 'preprocessing_params.json')) as f:
    pp = json.load(f)
TARGET_MEAN = pp['target_mean']
TARGET_STD  = pp['target_std']

with open(os.path.join(MODEL_DIR, 'hp_study_results.json')) as f:
    hp_results = json.load(f)

print(f"Best HPs from training: {hp_results['best_params']}")
print(f"Val metrics: {hp_results['val_metrics_mwh']}")
print(f"Target denorm: z * {TARGET_STD:.4f} + {TARGET_MEAN:.4f}")

## Step 1 — Reconstruct dataset for test split

In [ ]:
# ── Load all three splits ──
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_processed.csv'),
                        parse_dates=['datetime_utc'], index_col='datetime_utc').reset_index()
val_df   = pd.read_csv(os.path.join(DATA_DIR, 'val_processed.csv'),
                        parse_dates=['datetime_utc'], index_col='datetime_utc').reset_index()
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test_processed.csv'),
                        parse_dates=['datetime_utc'], index_col='datetime_utc').reset_index()

# ── Continuous time_idx across all three splits ──
train_df['time_idx'] = np.arange(len(train_df))
val_df['time_idx']   = np.arange(len(train_df), len(train_df) + len(val_df))
test_df['time_idx']  = np.arange(len(train_df) + len(val_df),
                                  len(train_df) + len(val_df) + len(test_df))

train_df['group_id'] = '0'
val_df['group_id']   = '0'
test_df['group_id']  = '0'

print(f"Train: {len(train_df):,} rows  time_idx [0, {train_df['time_idx'].max()}]")
print(f"Val:   {len(val_df):,} rows   time_idx [{val_df['time_idx'].min()}, {val_df['time_idx'].max()}]")
print(f"Test:  {len(test_df):,} rows  time_idx [{test_df['time_idx'].min()}, {test_df['time_idx'].max()}]")

In [ ]:
# ── Reconstruct training dataset (needed for from_dataset) ──
training_dataset = TimeSeriesDataSet(
    train_df,
    time_idx='time_idx',
    target=TARGET,
    group_ids=['group_id'],
    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    min_encoder_length=MAX_ENCODER_LENGTH,
    time_varying_known_reals=KNOWN_FUTURE,
    time_varying_unknown_reals=OBSERVED,
    target_normalizer=None,
    scalers={col: None for col in KNOWN_FUTURE + OBSERVED},
    add_relative_time_idx=True,
    add_encoder_length=True,
    allow_missing_timesteps=False,
)

# ── Test dataset with encoder context from val tail ──
test_with_context = pd.concat([
    val_df.iloc[-MAX_ENCODER_LENGTH:],
    test_df
])

test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    test_with_context,
    predict=True,
    stop_randomization=True,
)

test_dataloader = test_dataset.to_dataloader(
    train=False, batch_size=64, num_workers=2
)

print(f"Test dataset: {len(test_dataset)} samples")

## Step 2 — Load model & generate predictions

In [ ]:
def denormalize(z_values):
    """Convert z-score normalized values back to original MWh scale."""
    return z_values * TARGET_STD + TARGET_MEAN

def compute_metrics(actual_mwh, predicted_mwh):
    """Compute RMSE, MAE, R² on original MWh scale."""
    residuals = actual_mwh - predicted_mwh
    mae  = np.abs(residuals).mean()
    rmse = np.sqrt((residuals ** 2).mean())
    ss_res = (residuals ** 2).sum()
    ss_tot = ((actual_mwh - actual_mwh.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot
    return {'MAE_MWh': float(mae), 'RMSE_MWh': float(rmse), 'R2': float(r2)}

In [ ]:
# Load best model checkpoint
ckpt_path = os.path.join(MODEL_DIR, 'tft_best.ckpt')
best_model = TemporalFusionTransformer.load_from_checkpoint(ckpt_path)
best_model.eval()
print(f"Loaded model from {ckpt_path}")
print(f"Model parameters: {sum(p.numel() for p in best_model.parameters()):,}")

In [ ]:
# Generate predictions
test_predictions = best_model.predict(
    test_dataloader, mode='prediction', return_x=True
)
test_raw = best_model.predict(
    test_dataloader, mode='raw', return_x=True
)

# Extract and denormalize
pred_z   = test_predictions.output.cpu().numpy()   # (n_windows, 24)
actual_z = test_predictions.y[0].cpu().numpy()      # (n_windows, 24)

pred_z_flat   = pred_z.flatten()
actual_z_flat = actual_z.flatten()

pred_mwh   = np.clip(denormalize(pred_z_flat), 0, None)
actual_mwh = denormalize(actual_z_flat)

print(f"Prediction windows: {pred_z.shape[0]}")
print(f"Total test points: {len(pred_z_flat)}")

## Step 3 — Test metrics

This is the one and only test set evaluation. No model selection decisions are made after this point.

In [ ]:
test_metrics = compute_metrics(actual_mwh, pred_mwh)

# ── MASE: Mean Absolute Scaled Error ──
# Naive baseline: persistence (same hour 24h ago)
test_gen = denormalize(test_df[TARGET].values)
persistence_mae = np.abs(test_gen[24:] - test_gen[:-24]).mean()
mase = test_metrics['MAE_MWh'] / persistence_mae

print("=" * 60)
print("TEST SET METRICS (original MWh scale)")
print("=" * 60)
print(f"  RMSE:  {test_metrics['RMSE_MWh']:.2f} MWh")
print(f"  MAE:   {test_metrics['MAE_MWh']:.2f} MWh")
print(f"  R²:    {test_metrics['R2']:.4f}")
print(f"  MASE:  {mase:.4f} (< 1 means TFT beats 24h persistence)")
print("=" * 60)

In [ ]:
# ── Per-horizon metrics ──
horizon_metrics = []
for h in range(MAX_PREDICTION_LENGTH):
    pred_h   = np.clip(denormalize(pred_z[:, h]), 0, None)
    actual_h = denormalize(actual_z[:, h])
    m = compute_metrics(actual_h, pred_h)
    m['horizon_h'] = h + 1
    horizon_metrics.append(m)

horizon_df = pd.DataFrame(horizon_metrics)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ['MAE_MWh', 'RMSE_MWh', 'R2']):
    ax.plot(horizon_df['horizon_h'], horizon_df[metric], marker='o', markersize=3)
    ax.set_xlabel('Forecast Horizon (hours)')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} by Forecast Horizon (Test Set)')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(horizon_df.to_string(index=False))

## Step 4 — Baseline comparisons

Compare the TFT against two simple baselines:

- **Persistence (24h):** predict the same hour from 24 hours ago
- **Climatological (hourly):** predict the average generation for each hour-of-day from the training set

In [ ]:
# ── Persistence baseline: same hour 24h ago ──
test_gen = denormalize(test_df[TARGET].values)
persistence_pred   = test_gen[:-24]
persistence_actual = test_gen[24:]
persistence_metrics = compute_metrics(persistence_actual, persistence_pred)

# ── Climatological baseline: hourly mean from training set ──
train_gen_mwh = denormalize(train_df[TARGET].values)
train_hours   = pd.to_datetime(train_df['datetime_utc']).dt.hour
hourly_means  = pd.Series(train_gen_mwh, index=train_hours.values).groupby(level=0).mean()

test_hours = pd.to_datetime(test_df['datetime_utc']).dt.hour.values
clim_pred  = hourly_means.loc[test_hours].values
clim_actual = test_gen
clim_metrics = compute_metrics(clim_actual, clim_pred)

# ── Comparison table ──
print(f"\n{'Model':<25s} {'RMSE (MWh)':>12s} {'MAE (MWh)':>12s} {'R²':>8s}")
print("-" * 60)
print(f"{'TFT (ours)':<25s} {test_metrics['RMSE_MWh']:12.2f} {test_metrics['MAE_MWh']:12.2f} {test_metrics['R2']:8.4f}")
print(f"{'Persistence (24h)':<25s} {persistence_metrics['RMSE_MWh']:12.2f} {persistence_metrics['MAE_MWh']:12.2f} {persistence_metrics['R2']:8.4f}")
print(f"{'Climatological (hourly)':<25s} {clim_metrics['RMSE_MWh']:12.2f} {clim_metrics['MAE_MWh']:12.2f} {clim_metrics['R2']:8.4f}")

## Step 5 — Visualizations

In [ ]:
# ── 5a: Time series — selected test weeks ──
# Pick representative weeks: one summer (high gen), one autumn (variable)
test_datetime = pd.to_datetime(test_df['datetime_utc'])

week_starts = [
    ('Summer peak (Aug 4-10, 2025)', '2025-08-04', '2025-08-10'),
    ('Autumn transition (Nov 3-9, 2025)', '2025-11-03', '2025-11-09'),
]

fig, axes = plt.subplots(len(week_starts), 1, figsize=(14, 4 * len(week_starts)))
if len(week_starts) == 1:
    axes = [axes]

for ax, (title, start, end) in zip(axes, week_starts):
    mask = (test_datetime >= start) & (test_datetime <= end + ' 23:00:00')
    test_week = test_df[mask.values]
    dt = pd.to_datetime(test_week['datetime_utc'])
    actual_week = denormalize(test_week[TARGET].values)

    ax.plot(dt, actual_week, 'k-', linewidth=1, label='Actual', alpha=0.8)
    ax.set_xlabel('Date')
    ax.set_ylabel('Generation (MWh)')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── 5b: Scatter plot — predicted vs actual ──
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(actual_mwh, pred_mwh, alpha=0.05, s=5, c='steelblue')
max_val = max(actual_mwh.max(), pred_mwh.max())
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1, label='Perfect forecast')
ax.set_xlabel('Actual Generation (MWh)')
ax.set_ylabel('Predicted Generation (MWh)')
ax.set_title('TFT: Predicted vs Actual (Test Set)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, max_val * 1.05)
ax.set_ylim(0, max_val * 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5c: Residual analysis ──
residuals = actual_mwh - pred_mwh

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(residuals, bins=100, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Residual (MWh)')
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'Residual Distribution (mean={residuals.mean():.1f}, std={residuals.std():.1f})')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(np.arange(len(residuals)), residuals, s=1, alpha=0.1, c='steelblue')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Prediction Index')
axes[1].set_ylabel('Residual (MWh)')
axes[1].set_title('Residuals Over Time')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── 5d: TFT interpretability — variable importance ──
interpretation = best_model.interpret_output(test_raw.output, reduction='sum')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Encoder variable importance
enc_vars = training_dataset.reals
enc_imp  = interpretation['encoder_variables'].cpu().numpy()
enc_sorted = sorted(zip(enc_vars, enc_imp), key=lambda x: x[1], reverse=True)
enc_names, enc_vals = zip(*enc_sorted)
axes[0].barh(range(len(enc_names)), enc_vals, color='steelblue')
axes[0].set_yticks(range(len(enc_names)))
axes[0].set_yticklabels(enc_names)
axes[0].set_xlabel('Importance')
axes[0].set_title('Encoder Variable Importance')
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3)

# Decoder variable importance
dec_imp  = interpretation['decoder_variables'].cpu().numpy()
dec_sorted = sorted(zip(enc_vars, dec_imp), key=lambda x: x[1], reverse=True)
dec_names, dec_vals = zip(*dec_sorted)
axes[1].barh(range(len(dec_names)), dec_vals, color='coral')
axes[1].set_yticks(range(len(dec_names)))
axes[1].set_yticklabels(dec_names)
axes[1].set_xlabel('Importance')
axes[1].set_title('Decoder Variable Importance')
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── 5e: Temporal attention pattern ──
attention = interpretation['attention'].cpu().numpy()  # (prediction_length, encoder_length)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(attention, aspect='auto', cmap='viridis',
               extent=[0, MAX_ENCODER_LENGTH, MAX_PREDICTION_LENGTH, 0])
ax.set_xlabel('Encoder Position (hours into the past)')
ax.set_ylabel('Prediction Horizon (hours ahead)')
ax.set_title('Temporal Attention Weights')
plt.colorbar(im, ax=ax, label='Attention Weight')
plt.tight_layout()
plt.show()

## Step 6 — Summary

In [ ]:
print("=" * 70)
print("FINAL TEST RESULTS — Cáceres TFT Solar Forecast")
print("=" * 70)

print(f"\nModel: Temporal Fusion Transformer")
print(f"Test period: 2025-07-01 to 2025-12-31")
print(f"Encoder length: {MAX_ENCODER_LENGTH}h ({MAX_ENCODER_LENGTH//24} days)")
print(f"Prediction length: {MAX_PREDICTION_LENGTH}h (day-ahead)")

print(f"\nHyperparameters (selected via Optuna, {hp_results['n_trials']} trials):")
for k, v in hp_results['best_params'].items():
    print(f"  {k}: {v}")

print(f"\n{'Metric':<15s} {'Validation':>12s} {'Test':>12s}")
print("-" * 42)
for metric in ['RMSE_MWh', 'MAE_MWh', 'R2']:
    val_v  = hp_results['val_metrics_mwh'][metric]
    test_v = test_metrics[metric]
    fmt = '.2f' if 'MWh' in metric else '.4f'
    print(f"{metric:<15s} {val_v:12{fmt}} {test_v:12{fmt}}")
print(f"{'MASE':<15s} {'—':>12s} {mase:12.4f}")

print(f"\nBaseline comparison (test set):")
print(f"  TFT R²:          {test_metrics['R2']:.4f}")
print(f"  Persistence R²:  {persistence_metrics['R2']:.4f}")
print(f"  Climatology R²:  {clim_metrics['R2']:.4f}")

print(f"\nMASE = {mase:.4f} → {'TFT outperforms persistence' if mase < 1 else 'Persistence is competitive'}")
print("=" * 70)